# Hotel Booking Cancellation — EDA & Modeling

**Project:** Classical ML Pipeline  
**Dataset:** Hotel Booking Demand (Kaggle)  
**Author:** Sai  
**GitHub:** https://github.com/karthi9407/hotel-booking-cancellation

---

## Problem Framing

**Prediction task:** Binary classification — will this booking be cancelled?  
**Prediction point:** The moment a booking is confirmed. We can only use information available at that instant.  
**Target:** `is_canceled` (1 = cancelled, 0 = not cancelled)  
**Implication:** Any feature generated after booking confirmation is leakage and must be dropped.

## Session 1 — Data Load & First Look

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    classification_report, roc_auc_score,
    precision_recall_curve, roc_curve
)
from xgboost import XGBClassifier

%matplotlib inline
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

In [ ]:
df = pd.read_csv('../data/raw/hotel_bookings.csv')
print(f'Shape: {df.shape}')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

### Session 1 — Observations

1. The data covers arrivals between 2015 and 2017
2. Some fields are binary flags (is_canceled, is_repeated_guest) stored as int64
3. There are only two hotel types: City Hotel and Resort Hotel
4. Company and agent columns have substantial missing values — this is signal, not noise
5. The days_in_waiting_list column has extreme outliers (max 391 days)

---
## Session 2 — EDA Part 1: Univariate Analysis

In [ ]:
# Missing data audit
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({'missing': missing, 'pct': missing_pct})
missing_df[missing_df['missing'] > 0].sort_values('pct', ascending=False)

In [ ]:
# Class balance
print(df['is_canceled'].value_counts())
print(df['is_canceled'].value_counts(normalize=True).round(3))

In [ ]:
# Numeric distributions
numeric_cols = df.select_dtypes(include='number').columns.drop('is_canceled')
df[numeric_cols].hist(figsize=(18, 12), bins=40)
plt.tight_layout()
plt.show()

In [ ]:
# Categorical distributions
cat_cols = df.select_dtypes(include='object').columns
fig, axes = plt.subplots(len(cat_cols), 1, figsize=(10, 4 * len(cat_cols)))
for i, col in enumerate(cat_cols):
    df[col].value_counts().plot(kind='bar', ax=axes[i], title=col)
    axes[i].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Suspicious values
print("adr negatives:", (df['adr'] < 0).sum())
print("lead_time > 365:", (df['lead_time'] > 365).sum())
print("adults == 0:", (df['adults'] == 0).sum())
print("stays both 0:", ((df['stays_in_weekend_nights'] == 0) & (df['stays_in_week_nights'] == 0)).sum())

# Inspect adult=0 rows
df[df['adults'] == 0][['adults', 'children', 'babies', 'is_canceled']].head(10)

### Session 2 — Summary

The dataset covers 119,390 hotel bookings across two properties (Resort and City Hotel) from 2015–2017. 37% of bookings were cancelled, making accuracy a misleading metric. Four columns have missing values: company (94%) and agent (14%) are missing by design — those are direct bookings — and their missingness is itself a signal worth encoding as a feature. A handful of suspicious records exist: one negative ADR, 715 zero-night stays, and 403 adult-free bookings, all of which will be dropped before modeling. Two columns — deposit_type and distribution_channel — are heavily skewed toward a single value but may still carry signal in their minority categories.

---
## Session 3 — EDA Part 2: Bivariate Analysis

In [ ]:
def cancellation_rate(df, col):
    return (df.groupby(col)['is_canceled']
              .agg(['mean', 'count'])
              .rename(columns={'mean': 'cancel_rate', 'count': 'n'})
              .sort_values('cancel_rate', ascending=False)
              .round(3))

In [ ]:
# Lead time vs cancellation
fig, ax = plt.subplots(figsize=(10, 4))
df.groupby(pd.cut(df['lead_time'], bins=[0,7,30,90,180,365,740]))['is_canceled'].mean().plot(kind='bar', ax=ax)
ax.set_title('Cancellation rate by lead time bucket')
ax.set_ylabel('Cancellation rate')
plt.xticks(rotation=30)
plt.show()

In [ ]:
# Key categoricals
for col in ['hotel', 'market_segment', 'distribution_channel', 'deposit_type', 'customer_type']:
    print(f"\n--- {col} ---")
    print(cancellation_rate(df, col))

In [ ]:
# Month seasonality
month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
rates = df.groupby('arrival_date_month')['is_canceled'].mean().reindex(month_order)
rates.plot(kind='bar', figsize=(10, 4), title='Cancellation rate by arrival month')
plt.ylabel('Cancellation rate')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Repeated guest and previous behaviour
for col in ['is_repeated_guest', 'previous_cancellations', 'total_of_special_requests']:
    print(f"\n--- {col} ---")
    print(cancellation_rate(df, col))

In [ ]:
# Room type mismatch — NOTE: this is leakage (assigned at check-in, not at booking)
df['room_mismatch'] = (df['reserved_room_type'] != df['assigned_room_type']).astype(int)
print(cancellation_rate(df, 'room_mismatch'))

In [ ]:
# Leakage audit — columns to drop before modeling
leakage_cols = [
    'reservation_status',       # directly encodes the outcome
    'reservation_status_date',  # set after cancellation
    'assigned_room_type',       # known only at check-in
]
print("Remaining columns after leakage drop:")
print([c for c in df.columns if c not in leakage_cols])

### Session 3 — Summary

Key findings: longer lead time strongly predicts cancellation; deposit_type Non Refund has ~99% cancellation rate (counterintuitive — driven by OTA bulk bookings); room_mismatch appears predictive but is leakage (assigned room is only known at check-in). Three columns dropped before modeling: reservation_status, reservation_status_date, and assigned_room_type.

---
## Session 4 — Train/Val/Test Split & Baseline

In [ ]:
# Drop leakage and clean suspicious rows
leakage_cols = ['reservation_status', 'reservation_status_date', 'assigned_room_type']
df_clean = df.drop(columns=leakage_cols)

df_clean = df_clean[~((df_clean['stays_in_weekend_nights'] == 0) &
                       (df_clean['stays_in_week_nights'] == 0))]
df_clean = df_clean[df_clean['adults'] > 0]
df_clean = df_clean[df_clean['adr'] >= 0]

print(f"Rows after cleaning: {len(df_clean)} (dropped {len(df) - len(df_clean)})")

In [ ]:
# Time-based split — train: 2015-2016, val: Jan-May 2017, test: Jun-Aug 2017
df_clean['arrival_date'] = pd.to_datetime(
    df_clean['arrival_date_year'].astype(str) + '-' +
    df_clean['arrival_date_month'] + '-' +
    df_clean['arrival_date_day_of_month'].astype(str),
    format='%Y-%B-%d'
)

train = df_clean[df_clean['arrival_date'] < '2017-01-01']
val   = df_clean[(df_clean['arrival_date'] >= '2017-01-01') &
                 (df_clean['arrival_date'] < '2017-06-01')]
test  = df_clean[df_clean['arrival_date'] >= '2017-06-01']

print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")
print(f"Train cancel rate: {train['is_canceled'].mean():.3f}")
print(f"Val cancel rate:   {val['is_canceled'].mean():.3f}")
print(f"Test cancel rate:  {test['is_canceled'].mean():.3f}")

In [ ]:
# Baseline features (numeric only)
target = 'is_canceled'
baseline_features = [
    'lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights',
    'adults', 'is_repeated_guest', 'previous_cancellations',
    'previous_bookings_not_canceled', 'booking_changes',
    'days_in_waiting_list', 'adr', 'total_of_special_requests'
]

X_train_b = train[baseline_features]
y_train_b = train[target]
X_val_b   = val[baseline_features]
y_val_b   = val[target]

In [ ]:
# Baseline 1: majority class
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train_b, y_train_b)
y_pred_dummy = dummy.predict(X_val_b)

print("=== Baseline: Majority Class ===")
print(classification_report(y_val_b, y_pred_dummy))
print(f"ROC AUC: {roc_auc_score(y_val_b, dummy.predict_proba(X_val_b)[:,1]):.3f}")

In [ ]:
# Baseline 2: logistic regression (numeric only)
lr_baseline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
])

lr_baseline.fit(X_train_b, y_train_b)
y_pred_lr = lr_baseline.predict(X_val_b)
y_prob_lr  = lr_baseline.predict_proba(X_val_b)[:,1]

print("=== Baseline: Logistic Regression (numeric only) ===")
print(classification_report(y_val_b, y_pred_lr))
print(f"ROC AUC: {roc_auc_score(y_val_b, y_prob_lr):.3f}")

### Session 4 — Summary

Time-based split used (train: 2015–2016, val: Jan–May 2017, test: Jun–Aug 2017) to reflect real deployment conditions. Majority class baseline scores 0.500 AUC — a coin flip. Logistic regression on numeric features only scores 0.730 AUC and catches only 36% of cancellations, giving a clear floor to beat in Session 5 with proper feature engineering.

---
## Session 5 — Feature Engineering

In [ ]:
df_model = df_clean.copy()

# Missingness as signal
df_model['is_corporate']     = df_model['company'].notnull().astype(int)
df_model['booked_via_agent'] = df_model['agent'].notnull().astype(int)

# Lead time buckets
df_model['lead_time_bucket'] = pd.cut(
    df_model['lead_time'],
    bins=[0, 7, 30, 90, 180, 365, 740],
    labels=['0-7d','1-4wk','1-3mo','3-6mo','6-12mo','12mo+']
).astype(str)

# Fill small missing values
df_model['children'] = df_model['children'].fillna(0)
df_model['country']  = df_model['country'].fillna('Unknown')

# Total guests
df_model['total_guests'] = df_model['adults'] + df_model['children'] + df_model['babies']

In [ ]:
numeric_features = [
    'lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights',
    'adults', 'total_guests', 'is_repeated_guest',
    'previous_cancellations', 'previous_bookings_not_canceled',
    'booking_changes', 'days_in_waiting_list', 'adr',
    'total_of_special_requests', 'is_corporate', 'booked_via_agent'
]

categorical_features = [
    'hotel', 'meal', 'market_segment', 'distribution_channel',
    'deposit_type', 'customer_type', 'lead_time_bucket'
]

train_m = df_model[df_model['arrival_date'] < '2017-01-01']
val_m   = df_model[(df_model['arrival_date'] >= '2017-01-01') &
                   (df_model['arrival_date'] < '2017-06-01')]
test_m  = df_model[df_model['arrival_date'] >= '2017-06-01']

X_train = train_m[numeric_features + categorical_features]
y_train = train_m['is_canceled']
X_val   = val_m[numeric_features + categorical_features]
y_val   = val_m['is_canceled']
X_test  = test_m[numeric_features + categorical_features]
y_test  = test_m['is_canceled']

In [ ]:
# sklearn Pipeline — fit on train only, applied to val/test
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

In [ ]:
# Logistic regression — full features, balanced weights
lr_full = Pipeline([
    ('preprocessor', preprocessor),
    ('lr', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

lr_full.fit(X_train, y_train)
y_pred_full = lr_full.predict(X_val)
y_prob_full = lr_full.predict_proba(X_val)[:,1]

print("=== Logistic Regression — Full Features ===")
print(classification_report(y_val, y_pred_full))
print(f"ROC AUC: {roc_auc_score(y_val, y_prob_full):.3f}")

---
## Session 6 — Logistic Regression Tuned

In [ ]:
param_grid = {'lr__C': [0.01, 0.1, 1.0, 10.0]}

grid_search = GridSearchCV(
    lr_full, param_grid, cv=3, scoring='roc_auc', n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

print(f"Best C: {grid_search.best_params_}")
print(f"Best CV AUC: {grid_search.best_score_:.3f}")

best_lr = grid_search.best_estimator_
y_prob_best_lr = best_lr.predict_proba(X_val)[:,1]
print(f"Val AUC: {roc_auc_score(y_val, y_prob_best_lr):.3f}")

In [ ]:
# Coefficient interpretation
ohe_features = (best_lr.named_steps['preprocessor']
                .named_transformers_['cat']
                .named_steps['onehot']
                .get_feature_names_out(categorical_features))

all_features = numeric_features + list(ohe_features)
coefficients = best_lr.named_steps['lr'].coef_[0]

coef_df = (pd.DataFrame({'feature': all_features, 'coefficient': coefficients})
             .sort_values('coefficient', ascending=False))

print("=== Top 10 features pushing toward CANCELLATION ===")
print(coef_df.head(10).to_string(index=False))

print("\n=== Top 10 features pushing toward NOT CANCELLED ===")
print(coef_df.tail(10).to_string(index=False))

---
## Session 7 — Random Forest

In [ ]:
rf = Pipeline([
    ('preprocessor', preprocessor),
    ('rf', RandomForestClassifier(
        n_estimators=300, class_weight='balanced',
        random_state=42, n_jobs=-1
    ))
])

rf.fit(X_train, y_train)
y_prob_rf = rf.predict_proba(X_val)[:,1]
y_pred_rf = rf.predict(X_val)

print("=== Random Forest ===")
print(classification_report(y_val, y_pred_rf))
print(f"ROC AUC: {roc_auc_score(y_val, y_prob_rf):.3f}")

In [ ]:
# Feature importance
ohe_features_rf = (rf.named_steps['preprocessor']
                   .named_transformers_['cat']
                   .named_steps['onehot']
                   .get_feature_names_out(categorical_features))

all_features_rf = numeric_features + list(ohe_features_rf)
importances = rf.named_steps['rf'].feature_importances_

imp_df = (pd.DataFrame({'feature': all_features_rf, 'importance': importances})
            .sort_values('importance', ascending=False)
            .head(15))

imp_df.plot(kind='barh', x='feature', y='importance', figsize=(8,6), legend=False)
plt.title('Random Forest — Top 15 Feature Importances')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

---
## Session 8 — XGBoost

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb = Pipeline([
    ('preprocessor', preprocessor),
    ('xgb', XGBClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        scale_pos_weight=scale_pos_weight,
        eval_metric='auc', random_state=42, n_jobs=-1
    ))
])

xgb.fit(X_train, y_train)
y_prob_xgb = xgb.predict_proba(X_val)[:,1]
y_pred_xgb = xgb.predict(X_val)

print("=== XGBoost ===")
print(classification_report(y_val, y_pred_xgb))
print(f"ROC AUC: {roc_auc_score(y_val, y_prob_xgb):.3f}")

---
## Session 9 — Threshold Tuning & Final Evaluation

In [ ]:
# Find best threshold on val set
precisions, recalls, thresholds = precision_recall_curve(y_val, y_prob_xgb)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_idx = f1_scores.argmax()
best_threshold = thresholds[best_idx]

print(f"Best threshold (max F1): {best_threshold:.3f}")
print(f"Precision at best: {precisions[best_idx]:.3f}")
print(f"Recall at best:    {recalls[best_idx]:.3f}")
print(f"F1 at best:        {f1_scores[best_idx]:.3f}")

In [ ]:
# ROC and PR curves
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

fpr, tpr, _ = roc_curve(y_val, y_prob_xgb)
axes[0].plot(fpr, tpr, label=f'XGBoost (AUC={roc_auc_score(y_val, y_prob_xgb):.3f})')
axes[0].plot([0,1],[0,1],'--', label='Random (AUC=0.500)')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()

axes[1].plot(recalls, precisions, label='XGBoost')
axes[1].axvline(recalls[best_idx], color='red', linestyle='--', label=f'Best threshold ({best_threshold:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# FINAL TEST SET EVALUATION — run once, no going back
y_prob_test = xgb.predict_proba(X_test)[:,1]
y_pred_test = (y_prob_test >= best_threshold).astype(int)

print("=== FINAL TEST SET EVALUATION ===")
print(classification_report(y_test, y_pred_test))
print(f"ROC AUC: {roc_auc_score(y_test, y_prob_test):.3f}")

### Model Comparison

| Model | Val ROC AUC | Recall (cancelled) |
|---|---|---|
| Majority class | 0.500 | 0% |
| LR — numeric only | 0.730 | 36% |
| LR — full features, balanced | 0.820 | 69% |
| Random Forest | 0.810 | — |
| XGBoost (val, tuned threshold) | 0.835 | 79% |
| **XGBoost — final test** | **0.796** | **79%** |

See [WRITEUP.md](../WRITEUP.md) for full explanation of decisions and lessons.